# Private Health Supplements RAG

A fully private, local RAG pipeline for querying health supplement documents using **RAGWire**, **LangChain agents**, and a locally-hosted **Ollama** LLM.

## Features

- **Document Ingestion** — Ingest a directory of health supplement docs with automatic metadata extraction
- **Semantic Retrieval** — Retrieve top-k relevant chunks using vector similarity search
- **Metadata Filtering** — Use `get_filter_context` to inspect available metadata fields and apply smart filters before searching
- **LangChain Agent** — Conversational agent with tool use (`search_documents`, `get_filter_context`) and persistent memory via `InMemorySaver`
- **Fully Private** — All inference runs locally via Ollama; no data leaves your machine

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from ragwire import RAGWire, setup_logging
import ragwire

logger = setup_logging(log_level="INFO")

print(ragwire.__version__)

rag = RAGWire("config.yaml")

# Ingest documents — custom fields extracted from each doc
stats = rag.ingest_directory("../health_data/")
print(f"Ingested {stats['processed']} docs, {stats['chunks_created']} chunks")


In [4]:
stats

{'total': 11,
 'processed': 10,
 'skipped': 1,
 'failed': 0,
 'chunks_created': 75,
 'errors': []}

In [7]:
rag.retrieve('protein')

2026-03-27 14:28:53,384 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: protein...


[Document(metadata={'source': '..\\health_data\\05_Protein_Muscle_Synthesis_2024.pdf', 'file_name': '05_Protein_Muscle_Synthesis_2024.pdf', 'file_type': 'pdf', 'file_hash': '42c37704cd658c330f0b7c0c0a3809fccda2b2d79c476e38c29f86713ef5a9b4', 'chunk_id': '42c37704cd658c330f0b7c0c0a3809fccda2b2d79c476e38c29f86713ef5a9b4_6', 'chunk_hash': '4de4cbcaf523d0aa7f6687de17aeb371c06d1ffe8923c14b029301a198f223b4', 'chunk_index': 6, 'total_chunks': 11, 'created_at': '2026-03-27T08:51:09.396625+00:00', 'title': 'ergogenic effects of supplement combinations on endurance performance: a systematic review and meta-analysis of randomized controlled trials', 'authors': ['sebastian zart', 'michael fröhlich'], 'publication_year': 2025, 'research_focus': ['ergogenic aids', 'performance-enhancing substances', 'synergistic effects', 'performance', 'endurance'], '_id': '3da9f24d-5ede-4df8-be16-38f03008fb36', '_collection_name': 'health-supplements-papers'}, page_content=')\n9\n1\n0\n2\n(\n\n.\nl\n\na\n\nt\ne\n\n

## Full Wroking Agent

In [8]:
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

from ragwire import RAGWire, setup_logging

logger = setup_logging(log_level="INFO")



# ------------------------------------------------------------------ #
# 2. Tools
# ------------------------------------------------------------------ #
@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.

    Skip this for purely semantic queries with no metadata intent.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=5, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        meta_parts = [
            f"{k}={str(v)[:100]}"
            for k, v in doc.metadata.items()
            if k != "file_name" and v not in (None, "", [])
        ]
        header = f"[{source}" + (f" | {', '.join(meta_parts)}" if meta_parts else "") + "]"
        chunks.append(f"{header}\n{doc.page_content}")

    return "\n\n---\n\n".join(chunks)


# ------------------------------------------------------------------ #
# 3. Agent with memory
# ------------------------------------------------------------------ #
model = ChatOllama(model="qwen3.5:9b", base_url="http://localhost:11434")
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful health and fitness assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering — never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "If no relevant documents are found, say so — do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)


In [12]:
from IPython.display import Markdown

config = {"configurable": {"thread_id": "demo"}}


# ------------------------------------------------------------------ #
# 4. Interactive Q&A loop
# ------------------------------------------------------------------ #
print("\nRAG Agent ready. Type 'quit' to exit.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        break
    if not question:
        continue

    response = agent.invoke(
        {"messages": [HumanMessage(question)]},
        config=config,
    )
    
    print(f"\n\n\nQuery: {question}\n\n")
    print(response['messages'][-1].text)


RAG Agent ready. Type 'quit' to exit.




Query: What is benefit of Caffine in Gym


Based on the search results from the research documents, here are the **benefits of caffeine** for gym workouts:

## **Key Benefits:**

| Benefit | Details |
|---------|---------|
| **Explosiveness** | Caffeine improves explosiveness and power output before training sessions |
| **Alertness** | Increases mental sharpness, focus, and coordination during workouts |
| **Ergogenic Effect** | Demonstrates consistent performance-enhancing effects when taken acutely before training |
| **Sprint Performance** | Improves high-intensity, short-duration activities |
| **Technical Skills** | Enhances precision activities (e.g., shooting, technique) |
| **Psychological Benefits** | Increases self-confidence and mental focus during demanding workouts |

## **Optimal Usage:**

| Parameter | Recommendation |
|-----------|----------------|
| **Timing** | 45-60 minutes before exercise |
| **Dosage** | 3-6 mg per kg of 